# astro-ph RSS feed sandbox

Sandbox to develop an RSS feed reader/parser for astro-ph for use by OSU AstroCoffee.

We use the Python `requests` and `BeautifulSoup` modules that come with the Anaconda python3 distribution to
retrieve and parse the astro-ph RSS feed XML data.'

This notebook is also useful for looking at the RSS feed directly when we see funny things in the AstroCoffee
webpages (things like no papers on one day, or a previous day's postings appearing).

The RSS feed at arXiv has been increasingly buggy.

In [6]:
import numpy as np
import time
import re

# XML parsing

import requests
from bs4 import BeautifulSoup

# long text wrapping

import textwrap
from datetime import datetime

# URL of the astro-ph RSS feed

rssFeed = 'http://arxiv.org/rss/astro-ph'

# Maximum number of authors to display before switching to "et al." and
# a co-author count

maxAuth = 10


## Request and parse the astro-ph RSS feed

Use `requests.get()` method to retrieve the feed.

Use `BeautifulSoup` to parse the feed text which is in XML

In [7]:
print('Requesting the astro-ph RSS feed...')
t0 = time.time()
r = requests.get(rssFeed)
dt = time.time() - t0
print(f'... done, request took {dt:.3f} seconds')

# parse it

print('parsing the feed...')
t0 = time.time()
soup = BeautifulSoup(r.text,'xml')
dt = time.time() - t0
print(f'... done, parsing took {dt:.3f} seconds')

Requesting the astro-ph RSS feed...
... done, request took 0.184 seconds
parsing the feed...
... done, parsing took 0.573 seconds


### What is the publication date of this RSS feed?

The publication date of the RSS feed is in the `pubDate` XML tag.

We use this data to make two strings:
 * `postingDate` - a human-readable 'Year Month Day' string for display
 * `archiveDate` - a concatenated `CCYYMonDD` format string to use as the unique rootname for the local archive copy
 
### Diagnosis of problems

This is effectively the date the RSS feed XML file was created.  

If the date does not correspond to the present day, this can be what happens when the RSS feed was
not updated.

A different failure modes is for a new day's RSS feed to contain papers from a previous day, but the `pubDate`
tag is correct.  This means the value of the `pubDate` tag is not bulletproof.


In [8]:
pubDate = soup.find_all('pubDate')
pubBits = pubDate[0].text.split(' ')
postingDate = f'{pubBits[3]} {pubBits[2]} {pubBits[1]}'
archiveDate = f'{pubBits[3]}{pubBits[2]}{pubBits[1]}'

print(f'astro-ph Posting date: {postingDate}')
print(f'Daily Brew archive file rootname: {archiveDate}')

astro-ph Posting date: 2026 Mar 10
Daily Brew archive file rootname: 2026Mar10


### Extract information on each paper in the current feed

Each paper is stored as an XML `item` tag.  For each paper, extract
 * `link` - the URL to the abstract on the arxiv.org website
 * `arxiv:announce_type` - the submission time (new, replace[ment],cross[-listing], replace-cross)
 * `title` - the text of the paper title
 * `description` - the full text of the abstract
 * `dc:creator` - the author list
 
The author list inside the `dc:creator` item is a series of links to the arxiv.org author search.  We just want
the text as a list. Extract the author names from the stack of links (strings of text separated by \n for each
individual `href` tag).  We then build the full author list plus a subset if there are more than `maxAuth` authors.

In [9]:
papers = soup.find_all('item')

numPapers = len(papers)

if numPapers > 1:
    titles = []
    links = []
    abstracts = []
    authors = []
    subTypes = []

    for paper in papers:
        titles.append(paper.find('title').text)
        links.append(paper.find('link').text)
        abstracts.append(paper.find('description').text)
        subTypes.append(paper.find('arxiv:announce_type').text)

        # author are in dc:creator, which can be a mess as arXiv user inputs
        # are not rigorously filtered. The extra processing below addresses
        # common issues we've encountered
        
        rawAuth = paper.find('dc:creator')
        pAuth = rawAuth.text.split('\n')[1:-1]
        pAuth = re.sub( "<\/?dc:creator>", "", rawAuth.text).split(',')
        if len(pAuth) <= maxAuth:
            for auth in pAuth:
                auth = auth.replace(",","")
                if auth == pAuth[0]:
                    authList = auth.strip()
                else:
                    authList = f'{authList}, {auth.strip()}'
        else:
            authList = f'{pAuth[0].replace(",","").strip()}'
            for i in range(1,maxAuth):
                authList = f'{authList}, {pAuth[i].replace(",","").strip()}'
            authList = f'{authList}, et al. [{len(pAuth)-1} co-authors]'

        authors.append(authList)

    # create numpy arrays with the data
    
    paperTitle = np.array(titles)
    paperLink = np.array(links)
    paperAuths = np.array(authors)
    paperType = np.array(subTypes)
    paperAbst = np.array(abstracts)


## Print a summary

Print sorted into new postings, cross-posted, and replacements and print a formatted summary of everything
in today's astro-ph RSS posting.

Use `textwrap.wrap()` to make the abstract text readable.

If we found no papers, say so,

In [10]:
if numPapers > 1:

    # Separate out new postings, cross-listings, and replacement (include cross-replace)
    
    iNew = np.where(paperType=='new')[0]
    iCross = np.where(paperType=='cross')[0]
    iReplace = np.where(paperType=='replace')[0]

    print(f'Found {numPapers} papers on astro-ph for {postingDate}:\n{len(iNew)} new, {len(iReplace)} replacements, {len(iCross)} cross-posted')

    # Formatted summary of papers found (link, type, title, authors, abstract)
    
    print(f'\nNew Papers:')
    for i in iNew:
        print(f'\n{paperLink[i]} [{paperType[i]}]')
        print(f'  Title:   {paperTitle[i]}')
        print(f'  Authors: {paperAuths[i]}\n')
        for line in textwrap.wrap(paperAbst[i]):
            print(f'  {line}')
    
    print(f'\nCross Posts:')
    for i in iCross:
        print(f'\n{paperLink[i]} [{paperType[i]}]')
        print(f'  Title:   {paperTitle[i]}')
        print(f'  Authors: {paperAuths[i]}\n')
        for line in textwrap.wrap(paperAbst[i]):
            print(f'  {line}')
    
    print(f'\nReplacements:')
    for i in iReplace:
        print(f'\n{paperLink[i]} [{paperType[i]}]')
        print(f'  Title:   {paperTitle[i]}')
        print(f'  Authors: {paperAuths[i]}\n')
        for line in textwrap.wrap(paperAbst[i]):
            print(f'  {line}')

else:
    print(f'No papers found in the {postingDate} astro-ph RSS feed')


Found 122 papers on astro-ph for 2026 Mar 10:
73 new, 27 replacements, 12 cross-posted

New Papers:

https://arxiv.org/abs/2603.05571 [new]
  Title:   UK White Paper on Magnetohydrodynamic (MHD) seismology of solar and heliospheric plasmas
  Authors: Valery M. Nakariakov, David B. Jess, Andrew N. Wright, Timothy K. Yeoman, Thomas Elsden, James A. McLaughlin, Dmitrii Y. Kolotkov, Viktor Fedun, Robertus Erd\'elyi

  arXiv:2603.05571v1 Announce Type: new  Abstract: Magnetohydrodynamic
  (MHD) seismology uses naturally occurring MHD waves to infer plasma
  properties that are otherwise hard to measure, especially magnetic
  field strength and topology, electric currents, fine structuring,
  transport coefficients, and energy release. Across the solar
  atmosphere, heliosphere, and planetary magnetospheres, multi-
  wavelength remote sensing and in-situ observations of waves provide
  powerful diagnostics that can address major open problems including
  chromospheric and coronal heating, fl